<a href="https://colab.research.google.com/github/KyberNull/embeddinggemma2_asr/blob/main/DSP_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmbeddingGemma 2 based ASR

This notebook is a **TPU-oriented baseline** for the project:

- EmbeddingGemma 2 audio encoder, frozen.
- Extract temporally ordered intermediate audio hidden states.
- Learn a softmax layer mixture.
- LayerNorm + projection + causal depthwise Conv1D.
- Bidirectional GRU reader.
- Grapheme-level CTC output.

## Important scope

The proposed Mamba reader is **not used in this notebook**. The common Mamba implementations rely on custom CUDA kernels and are not a drop-in TPU implementation. This notebook uses a standard PyTorch GRU reader as a TPU-compatible baseline. Once the data pipeline and CTC objective work, replace the GRU with a TPU-native Mamba/SSM implementation and benchmark it.

The notebook also computes CTC loss on the CPU as a conservative compatibility path. The reader forward/backward and optimizer run on the TPU; copying logits to CPU for CTC loss is slower than a native TPU loss, but avoids assuming that the installed TPU backend implements the CTC kernel.

## Colab setup

1. Open **Runtime → Change runtime type → Hardware accelerator → TPU**.
2. Run cells from top to bottom.
3. Put your dataset in Google Drive and create a CSV manifest with columns `path,text`.
4. Use mono audio. The notebook resamples to 16 kHz.
5. Start with a small subset (e.g. 100–500 utterances) to validate the pipeline.

Free Colab accelerator availability and usage limits are not guaranteed. A TPU runtime may not be offered to every session.

## 1. Install dependencies

This uses the current PyTorch-native `torch_tpu` backend. PyTorch/XLA is the older integration; new PyTorch TPU work is moving to TorchTPU.

If installation fails because the Colab image and wheel versions are incompatible, do not randomly mix PyTorch/XLA and TorchTPU in the same runtime. Restart the runtime and use the installation instructions linked in the notes at the end.

In [1]:
!pip -q install torch_tpu polars
!pip -q install -U "transformers>=5.0.0" librosa soundfile tqdm jiwer regex

In [2]:
!pip -q install datasets

## 2. Verify that Colab actually assigned a TPU

Do not continue if this cell reports that the TPU is unavailable. Recheck the runtime type first.

In [3]:
import torch, sys, platform

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("TPU available:", torch.accelerator.is_available())
print("TPU device count:", torch.accelerator.device_count())

if not torch.accelerator.is_available():
    raise RuntimeError(
        "No TPU detected. In Colab choose Runtime > Change runtime type > TPU, "
        "then restart and rerun the notebook."
    )

DEVICE = torch.device("tpu")
x = torch.randn(128, 128, device=DEVICE)
y = x @ x
torch.accelerator.synchronize()
print("TPU smoke test:", y.device, tuple(y.shape))

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.14.1+cpu
TPU available: True
TPU device count: 1
TPU smoke test: tpu:0 (128, 128)


## 3. Mount Drive and configure the manifest

Your CSV must have:

| path | text |
|---|---|
| `/content/drive/MyDrive/asr/audio/utt001.wav` | transcript for utterance 1 |
| `/content/drive/MyDrive/asr/audio/utt002.wav` | transcript for utterance 2 |

`path` can be absolute or relative to `AUDIO_ROOT`.

Use a train/validation split at the **speaker or recording level** if you have speaker IDs. A random utterance split can leak speakers and recording conditions across the two sets.

In [ ]:
from datasets import load_dataset
import polars as pl
import os

CACHE_DIR = "/content/eg2_asr_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Streaming LibriSpeech (clean, 100h) from Hugging Face...")
ds = load_dataset("multimodalart/librispeech_asr_dummy", split="validation", streaming=False)

records = []
for item in ds:
    records.append({
        "path": item["file"],
        "text": item["text"].upper(),
        "audio_array": item["audio"]["array"],
        "sampling_rate": item["audio"]["sampling_rate"]
    })

df = pl.DataFrame(records)
print("Loaded utterances from Hugging Face dummy LibriSpeech:", len(df))
print(df.head())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, pandas as pd

MANIFEST = "/content/drive/MyDrive/asr/train.csv"   # CHANGE THIS
AUDIO_ROOT = "/content/drive/MyDrive/asr"           # CHANGE THIS
CACHE_DIR = "/content/eg2_asr_cache"                # local Colab disk; copy to Drive to persist
os.makedirs(CACHE_DIR, exist_ok=True)

df = pd.read_csv(MANIFEST)
assert {"path", "text"}.issubset(df.columns), "CSV must contain path,text columns"
df = df.dropna(subset=["path", "text"]).reset_index(drop=True)

def resolve_audio_path(p):
    p = str(p)
    return p if os.path.isabs(p) else os.path.join(AUDIO_ROOT, p)

df["path"] = df["path"].map(resolve_audio_path)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print("Utterances:", len(df))
display(df.head())

## 4. Load the audio processor and the audio-only EmbeddingGemma 2 model

The model card describes a 740M multimodal model, but the audio tower is a separately configurable component. We disable the text and vision towers to avoid loading them.

The processor runs on the host CPU. The audio encoder is placed on the TPU. `get_audio_features(..., output_hidden_states=True)` is the key call: we need the sequence of audio hidden states, not the pooled final embedding.

In [ ]:
from transformers import AutoProcessor, AutoModel
import torch

MODEL_ID = "google/embeddinggemma-2"

processor = AutoProcessor.from_pretrained(MODEL_ID)

encoder = AutoModel.from_pretrained(
    MODEL_ID,
    text_config=None,
    vision_config=None,
    dtype=torch.bfloat16,
    device_map="tpu",
)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)

print("Loaded model:", type(encoder).__name__)
print("Device:", next(encoder.parameters()).device)

## 5. Audio loading and hidden-state extraction

This caches one tensor per utterance with shape:

`[number_of_encoder_layers, time_steps, hidden_dim]`

The cache is stored as FP16 on local Colab storage. This lets the TPU train the reader without repeatedly running the 300M-parameter audio encoder.

For a first test, set `MAX_UTTERANCES = 100`. Increase it after checking that feature extraction and the CTC length constraints work.

In [ ]:
import librosa, numpy as np, torch
from tqdm.auto import tqdm
import polars as pl

SAMPLE_RATE = 16000
MAX_UTTERANCES = 100
rows = df.slice(0, MAX_UTTERANCES)

def load_audio_from_record(row):
    wav = row["audio_array"]
    sr = row["sampling_rate"]
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return np.asarray(wav, dtype=np.float32)

@torch.inference_mode()
def extract_hidden_states(wav):
    batch = processor(
        audio=[wav],
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )
    input_features = batch["input_features"].to(DEVICE)
    input_features_mask = batch.get("input_features_mask")
    if input_features_mask is not None:
        input_features_mask = input_features_mask.to(DEVICE)

    out = encoder.get_audio_features(
        input_features=input_features,
        input_features_mask=input_features_mask,
        output_hidden_states=True,
        return_dict=True,
    )

    hs = getattr(out, "hidden_states", None)
    if hs is None:
        hs = (out.last_hidden_state,)

    stacked = torch.stack([h[0] for h in hs], dim=0)
    return stacked.detach().to("cpu", dtype=torch.float16).contiguous()

cache_records = []
failed = []

for ridx, row in tqdm(enumerate(rows.to_dicts()), total=len(rows)):
    try:
        cache_path = os.path.join(CACHE_DIR, f"utt_{ridx:08d}.pt")
        if not os.path.exists(cache_path):
            wav = load_audio_from_record(row)
            feat = extract_hidden_states(wav)
            torch.save(feat, cache_path)
        else:
            feat = torch.load(cache_path, map_location="cpu", weights_only=True)

        cache_records.append({
            "cache_path": cache_path,
            "text": row["text"],
            "source_path": row["path"],
            "frames": int(feat.shape[1]),
            "layers": int(feat.shape[0]),
            "dim": int(feat.shape[2]),
        })
    except Exception as e:
        failed.append((row["path"], repr(e)))

cache_df = pl.DataFrame(cache_records)
print("Cached:", len(cache_df), "Failed:", len(failed))
if len(cache_df) > 0:
    print("Example feature shape:", torch.load(cache_df["cache_path"][0], map_location="cpu", weights_only=True).shape)
if failed[:5]:
    print("First failures:", failed[:5])

In [ ]:
import librosa, numpy as np, soundfile as sf, torch
from tqdm.auto import tqdm
from pathlib import Path

SAMPLE_RATE = 16000
MAX_UTTERANCES = 100  # set None to process the entire manifest
rows = df if MAX_UTTERANCES is None else df.iloc[:MAX_UTTERANCES]

def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=False)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return np.asarray(wav, dtype=np.float32)

@torch.inference_mode()
def extract_hidden_states(wav):
    batch = processor(
        audio=[wav],
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )
    input_features = batch["input_features"].to(DEVICE)
    input_features_mask = batch.get("input_features_mask")
    if input_features_mask is not None:
        input_features_mask = input_features_mask.to(DEVICE)

    out = encoder.get_audio_features(
        input_features=input_features,
        input_features_mask=input_features_mask,
        output_hidden_states=True,
        return_dict=True,
    )

    hs = getattr(out, "hidden_states", None)
    if hs is None:
        # Fallback: still trainable, but layer mixing becomes a one-layer mixture.
        hs = (out.last_hidden_state,)

    # [layers, batch=1, time, dim] -> [layers, time, dim]
    stacked = torch.stack([h[0] for h in hs], dim=0)
    return stacked.detach().to("cpu", dtype=torch.float16).contiguous()

cache_records = []
failed = []

for ridx, row in tqdm(rows.iterrows(), total=len(rows)):
    try:
        cache_path = os.path.join(CACHE_DIR, f"utt_{ridx:08d}.pt")
        if not os.path.exists(cache_path):
            wav = load_audio(row["path"])
            feat = extract_hidden_states(wav)
            torch.save(feat, cache_path)
        else:
            feat = torch.load(cache_path, map_location="cpu", weights_only=True)

        cache_records.append({
            "cache_path": cache_path,
            "text": row["text"],
            "source_path": row["path"],
            "frames": int(feat.shape[1]),
            "layers": int(feat.shape[0]),
            "dim": int(feat.shape[2]),
        })
    except Exception as e:
        failed.append((row["path"], repr(e)))

cache_df = pd.DataFrame(cache_records)
print("Cached:", len(cache_df), "Failed:", len(failed))
print("Example feature shape:", torch.load(cache_df.iloc[0].cache_path, map_location="cpu", weights_only=True).shape)
if failed[:5]:
    print("First failures:", failed[:5])

## 6. Build a grapheme-level vocabulary

For Indian scripts, code-point tokenization can split a visible grapheme into multiple Unicode code points. This notebook uses Unicode grapheme clusters (`regex`'s `\X`) after NFC normalization.

CTC blank is ID 0. Every grapheme, including space, gets a non-zero ID.

In [ ]:
import regex, unicodedata, json
from collections import Counter

def normalize_text(s):
    return unicodedata.normalize("NFC", str(s).strip())

def graphemes(s):
    return regex.findall(r"\X", normalize_text(s))

texts = [normalize_text(s) for s in cache_df["text"].to_list()]
counter = Counter(g for s in texts for g in graphemes(s))
itos = ["<blank>"] + sorted(counter.keys())
stoi = {tok: i for i, tok in enumerate(itos)}

print("Grapheme vocabulary size (including blank):", len(itos))
print("Example:", texts[0], "->", graphemes(texts[0]))

with open(os.path.join(CACHE_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(itos, f, ensure_ascii=False, indent=2)

## 7. Dataset and collate function

The collate function pads only the time dimension. Layer count and hidden dimension must be identical across examples.

The CTC feasibility condition is stricter than `T >= U` when adjacent target graphemes repeat. We therefore check:

`minimum_frames = target_length + number_of_adjacent_repeats`

Utterances that violate this condition are excluded and reported.

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import numpy as np

class CachedASRDataset(Dataset):
    def __init__(self, frame):
        self.items = []
        self.rejected = []
        for r in frame.to_dicts():
            toks = graphemes(r["text"])
            ids = [stoi[t] for t in toks]
            feat = torch.load(r["cache_path"], map_location="cpu", weights_only=True)
            repeats = sum(ids[i] == ids[i-1] for i in range(1, len(ids)))
            min_frames = len(ids) + repeats
            if feat.shape[1] < min_frames:
                self.rejected.append((r["source_path"], feat.shape[1], min_frames))
                continue
            self.items.append((r["cache_path"], torch.tensor(ids, dtype=torch.long), r["text"]))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y, text = self.items[i]
        return torch.load(p, map_location="cpu", weights_only=True).float(), y, text

def collate_fn(batch):
    feats, targets, texts = zip(*batch)
    B, L, D = len(feats), feats[0].shape[0], feats[0].shape[-1]
    maxT = max(x.shape[1] for x in feats)
    x = torch.zeros(B, L, maxT, D, dtype=torch.float32)
    input_lengths = torch.tensor([z.shape[1] for z in feats], dtype=torch.long)
    for i, z in enumerate(feats):
        x[i, :, :z.shape[1], :] = z.float()
    target_lengths = torch.tensor([len(y) for y in targets], dtype=torch.long)
    flat_targets = torch.cat(targets)
    return x, input_lengths, flat_targets, target_lengths, texts

dataset = CachedASRDataset(cache_df)
print("Usable:", len(dataset), "Rejected for CTC length:", len(dataset.rejected))
if dataset.rejected[:5]:
    print("First rejected:", dataset.rejected[:5])

n_val = max(1, int(0.05 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 8. TPU reader: learned layer mix + projection + causal depthwise convolution + BiGRU + CTC

This is the baseline reader. It intentionally avoids custom CUDA extensions.

- Layer mixing is trainable.
- The convolution is causal (left-padded only).
- The bidirectional GRU is an offline recognizer, not a streaming recognizer.
- Output time resolution is unchanged, so CTC uses the encoder frame lengths.

In [ ]:
import torch.nn as nn

class TPUCTCReader(nn.Module):
    def __init__(self, n_layers, input_dim, vocab_size, hidden=256, gru_layers=2, kernel_size=5):
        super().__init__()
        self.layer_logits = nn.Parameter(torch.zeros(n_layers))
        self.norm_in = nn.LayerNorm(input_dim)
        self.proj = nn.Linear(input_dim, hidden)

        self.dwconv = nn.Conv1d(
            hidden, hidden, kernel_size=kernel_size,
            groups=hidden, padding=0
        )
        self.conv_act = nn.GELU()

        self.gru = nn.GRU(
            input_size=hidden,
            hidden_size=hidden // 2,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.1 if gru_layers > 1 else 0.0,
        )
        self.norm_out = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, vocab_size)

        self.kernel_size = kernel_size

    def forward(self, x):
        # x: [B, layers, T, D]
        weights = torch.softmax(self.layer_logits, dim=0)
        x = torch.einsum("l,bltd->btd", weights, x)

        x = self.proj(self.norm_in(x))
        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))  # causal left padding
        x = self.dwconv(x)
        x = self.conv_act(x).transpose(1, 2)

        x, _ = self.gru(x)
        x = self.norm_out(x)
        logits = self.classifier(x)
        return logits.log_softmax(dim=-1)

sample_x, sample_il, sample_y, sample_tl, _ = next(iter(train_loader))
n_layers, _, input_dim = sample_x.shape[1], sample_x.shape[2], sample_x.shape[3]

reader = TPUCTCReader(
    n_layers=n_layers,
    input_dim=input_dim,
    vocab_size=len(itos),
).to(DEVICE)

with torch.no_grad():
    test_out = reader(sample_x.to(DEVICE))
print("Reader output:", test_out.shape, test_out.device)

## 9. Train with CTC loss

The reader and optimizer are on the TPU. For portability, this cell evaluates `CTCLoss` on CPU. The `.cpu()` copy remains part of the autograd graph, so gradients propagate back to the TPU reader, but this introduces transfer overhead.

For a production run, verify whether the installed TorchTPU version supports `CTCLoss` natively and move the loss to the TPU only after a forward/backward smoke test.

In [ ]:
import torch.optim as optim
from jiwer import cer
from tqdm.auto import tqdm

BLANK_ID = 0
ctc_loss_fn = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
optimizer = optim.AdamW(reader.parameters(), lr=2e-4, weight_decay=1e-4)

def train_one_epoch():
    reader.train()
    losses = []
    for x, input_lengths, targets, target_lengths, _ in tqdm(train_loader):
        x = x.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        log_probs = reader(x)
        log_probs_tbc = log_probs.transpose(0, 1)

        loss = ctc_loss_fn(
            log_probs_tbc.float().cpu(),
            targets,
            input_lengths,
            target_lengths,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reader.parameters(), 5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def validate():
    reader.eval()
    losses = []
    for x, input_lengths, targets, target_lengths, _ in val_loader:
        x = x.to(DEVICE)
        lp = reader(x).transpose(0, 1)
        loss = ctc_loss_fn(lp.float().cpu(), targets, input_lengths, target_lengths)
        losses.append(float(loss.cpu()))
    return float(np.mean(losses))

EPOCHS = 5
history = []
for epoch in range(EPOCHS):
    tr = train_one_epoch()
    va = validate()
    history.append({"epoch": epoch + 1, "train_ctc": tr, "val_ctc": va})
    print(history[-1])

    ckpt = {
        "model": {k: v.detach().cpu() for k, v in reader.state_dict().items()},
        "itos": itos,
        "input_dim": input_dim,
        "n_layers": n_layers,
        "epoch": epoch + 1,
        "history": history,
    }
    torch.save(ckpt, "/content/drive/MyDrive/eg2_ctc_reader.pt")

pl.DataFrame(history)

## 10. Greedy CTC decoding and a quick validation inspection

CTC greedy decoding takes the argmax at each frame, merges consecutive duplicate IDs, then removes blank. This is a baseline decoder; prefix beam search can improve accuracy.

In [ ]:
def ctc_greedy_decode(frame_ids):
    out = []
    prev = None
    for idx in frame_ids:
        idx = int(idx)
        if idx != prev and idx != BLANK_ID:
            out.append(itos[idx])
        prev = idx
    return "".join(out)

@torch.no_grad()
def inspect_predictions(n=10):
    reader.eval()
    shown = 0
    for x, input_lengths, targets, target_lengths, texts in val_loader:
        lp = reader(x.to(DEVICE)).cpu()
        pred = lp.argmax(dim=-1)
        for i in range(pred.shape[0]):
            hyp = ctc_greedy_decode(pred[i, :input_lengths[i]].tolist())
            print("REF:", texts[i])
            print("HYP:", hyp)
            print("-" * 60)
            shown += 1
            if shown >= n:
                return

inspect_predictions()

## 11. Save the feature cache to Drive

The feature cache is currently on ephemeral Colab storage. Copy it to Drive if you want to reuse it after the runtime disconnects.

In [ ]:
import shutil, os
PERSIST_CACHE = "/content/drive/MyDrive/eg2_asr_cache"
shutil.copytree(CACHE_DIR, PERSIST_CACHE, dirs_exist_ok=True)
print("Copied cache to:", PERSIST_CACHE)

## Notes and known limitations

1. **Mamba is not implemented here.** Standard Mamba CUDA kernels are not TPU kernels. The GRU is a practical TPU baseline, not a claim that it is equivalent to Mamba.
2. **The encoder path is version-sensitive.** This uses the Hugging Face `EmbeddingGemma2` API (`get_audio_features`, `output_hidden_states=True`). If the audio-only loading override is rejected by the installed Transformers version, load the full model and inspect its config/API before proceeding; do not silently train on the pooled 768-D embedding.
3. **CTC loss runs on CPU in this baseline.** This is correct but can become a bottleneck.
4. **Fixed shapes improve TPU compilation efficiency.** This notebook pads each batch to its longest sequence; for larger experiments, bucket utterances by duration and use fixed frame buckets.
5. **Colab Free is for prototyping.** Runtime availability, accelerator type, memory and quotas vary. It is not a dependable multi-day training cluster.
6. **The 100-utterance setting is only a smoke test.** It is not enough to make a useful ASR model.

### References

- [PyTorch on Cloud TPU / TorchTPU](https://docs.cloud.google.com/tpu/docs/pytorch-on-tpu)
- [Hugging Face: Google TPU](https://huggingface.co/docs/transformers/main/google_tpu)
- [EmbeddingGemma 2 Transformers documentation](https://huggingface.co/docs/transformers/model_doc/embedding_gemma2)
- [Gemma-ASR reference implementation](https://github.com/qnguyen3/gemma-asr)
- [sea-asr-ph model card and training recipe](https://huggingface.co/PixelML/sea-asr-ph)